In [ ]:
from kfp import dsl
import kfp
from kfp.dsl import Dataset, Input, Output
from kfp import compiler
from kfp import kubernetes


In [ ]:

@dsl.component(base_image="python:3.11", packages_to_install=["loguru"])
def get_paths_data(data_dir: str, paths_dataset: Output[Dataset]):
    import os
    import json
    from loguru import logger


    logger.debug(f"Data directory: {data_dir}")

    label_names = []
    for d in os.listdir(data_dir):
        if os.path.isdir(os.path.join(data_dir, d)):
            label_names.append(d)
    

    with open(paths_dataset.path, "w") as f:
        for label in label_names:
            label_dir = os.path.join(data_dir, label)
            for fname in os.listdir(label_dir):
                full_path = os.path.join(label_dir, fname)
                logger.debug(f"Processing file: {full_path} with label: {label}")
                record = {
                    "path": full_path,
                    "label": label
                }
                f.write(json.dumps(record) + "\n")

    logger.debug(f"Saved paths dataset to {paths_dataset.path}")


In [ ]:
@dsl.component(base_image="python:3.11", packages_to_install=["opencv-python-headless"])
def compute_image_stats(paths_dataset: Input[Dataset], stats_dataset: Output[Dataset]):
    import os
    import json
    import cv2

    def compute_stats(path, label) -> dict:

        img = cv2.imread(path)
        if img is None:
            return None

        stats = {}
        for i, color in enumerate(['Blue', 'Green', 'Red']):
            channel = img[:, :, i]
            stats[f'{color}_mean'] = float(channel.mean())
            stats[f'{color}_std'] = float(channel.std())
            stats[f'{color}_min'] = float(channel.min())
            stats[f'{color}_max'] = float(channel.max())

        img_num = os.path.basename(path).split("_")[1].split('.')[0]

        return {
            'image_id': img_num,
            'label': label,
            **stats
        }

    # Read paths and labels from previous component
    path_label_pairs = []
    with open(paths_dataset.path, "r") as f:
        for line in f:
            record = json.loads(line.strip())
            path_label_pairs.append((record["path"], record["label"]))

    results = []
    for path, label in path_label_pairs:
        result = compute_stats(path, label)
        if result is not None:
            results.append(result)

    # Filter out None (bad images)
    results = [r for r in results if r is not None]

    # Save as JSONL
    with open(stats_dataset.path, "w") as f:
        for record in results:
            f.write(json.dumps(record) + "\n")

    print(f"Processed {len(results)} images out of {len(path_label_pairs)}")

In [ ]:
@dsl.pipeline(name="ETL Pipeline")
def ETL_pipeline(data_dir: str):
    get_paths_task = get_paths_data(data_dir=data_dir)

    # Mount the PVC into the component pod
    get_paths_task.set_caching_options(False)
    kubernetes.mount_pvc(
        get_paths_task,
        pvc_name="kind-local-pvc",
        mount_path="/data/persistent",
    )

    stats_task = compute_image_stats(paths_dataset=get_paths_task.outputs["paths_dataset"])

    # Mount the PVC into the component pod
    stats_task.set_caching_options(False)
    kubernetes.mount_pvc(
        stats_task,
        pvc_name="kind-local-pvc",
        mount_path="/data/persistent",
    )


compiler.Compiler().compile(
    pipeline_func=ETL_pipeline,
    package_path="etl_pipeline.yaml",
)

client = kfp.Client("http://localhost:8888")

run = client.create_run_from_pipeline_package(
    pipeline_file='etl_pipeline.yaml',
    arguments={'data_dir': '/data/persistent/EuroSAT_RGB'},
    run_name='etl-run',
    experiment_name='etl-experiment',
)
print(run.run_id)

In [ ]:
from minio import Minio

client = Minio("localhost:9000", access_key="minio", secret_key="minio123", secure=False)

response = client.fget_object(
    bucket_name="mlpipeline",
    object_name="v2/artifacts/etl-pipeline/33302034-95a8-4311-a415-dc9d270efcd5/get-paths-data/0ae57d61-9b58-4a96-95ab-4410aa56e231/paths_dataset",
    file_path="./paths_dataset.jsonl"
)

print(response)

In [ ]:
import pandas as pd
df = pd.read_json("./paths_dataset.jsonl", lines=True)
print(df.head())